# Fake GeoTIFF result from legacy GPKG

Questa notebook cell ricostruisce un raster multi-banda uguale (strutturalmente) all'output della nuova versione dello script 4.2, partendo da un vecchio file GPKG con un punto al centro di ogni pixel e attributi per banda.

Output bands (in ordine):
1. `car_share`
2. `walk_share`
3. `best_reseller_id_walk`
4. `best_time_walk_min`
5. `best_reseller_id_car`
6. `best_time_car_min`

Puoi lanciare la cella sotto così com'è; se i path differiscono, modifica solo i parametri nella sezione `USER PARAMETERS`.

In [1]:
from __future__ import annotations

import numpy as np
import geopandas as gpd
import rasterio

# ============================================================
# USER PARAMETERS
# ============================================================
# Base folders: prova prima dataset_big, altrimenti datasetbig
DATA_DIR_CANDIDATES = ["dataset_big", "datasetbig"]
TRASH_SUBDIR = "trash"

# Legacy input: vecchio risultato a punti (1 punto = 1 pixel)
LEGACY_GPKG_NAME = "huff_preferred_distributor_per_pixel.gpkg"
LEGACY_LAYER_NAME: str | None = None  # None -> primo layer disponibile

# Reference grid (per avere stessa estensione/risoluzione/CRS dell'output nuovo)
REFERENCE_RASTER_NAME = "Population.tif"

# Output fake raster
OUTPUT_TIF_NAME = "huff_preferred_distributor_per_pixel_FAKE.tif"

# Nodata come script 4.2
NODATA_FLOAT = -9999.0

# Band order coerente con script 4.2
BAND_NAMES = [
    "car_share",
    "walk_share",
    "best_reseller_id_walk",
    "best_time_walk_min",
    "best_reseller_id_car",
    "best_time_car_min",
]


# ============================================================
# HELPERS
# ============================================================
def pick_data_dir(candidates: list[str]) -> str:
    for d in candidates:
        # Check the reference raster to confirm the directory is the right one
        ref_path = f"{d}/{REFERENCE_RASTER_NAME}"
        try:
            with rasterio.open(ref_path):
                return d
        except Exception:
            continue
    raise FileNotFoundError(
        "Nessuna cartella dati valida trovata. Cercati: "
        + ", ".join(candidates)
        + f" (con file {REFERENCE_RASTER_NAME})"
    )


def normalize_columns(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    # Remove accidental leading/trailing spaces in field names
    gdf = gdf.copy()
    gdf.columns = [str(c).strip() for c in gdf.columns]
    return gdf


def build_fake_raster_from_points(
    gpkg_path: str,
    ref_raster_path: str,
    output_tif_path: str,
    layer_name: str | None = None,
):
    # 1) Load legacy points
    if layer_name is None:
        layer_name = gpd.list_layers(gpkg_path).iloc[0]["name"]

    print(f"[1/5] Reading legacy points: {gpkg_path} | layer={layer_name}")
    gdf = gpd.read_file(gpkg_path, layer=layer_name)
    if gdf.empty:
        raise RuntimeError("Il layer legacy e' vuoto.")

    gdf = normalize_columns(gdf)
    gdf = gdf[gdf.geometry.notna()].copy()
    gdf = gdf[gdf.geometry.geom_type.isin(["Point"])].copy()
    if gdf.empty:
        raise RuntimeError("Nessuna geometria Point trovata nel GPKG legacy.")

    # 2) Open reference grid
    print(f"[2/5] Reading reference raster: {ref_raster_path}")
    with rasterio.open(ref_raster_path) as ref:
        ref_profile = ref.profile.copy()
        transform = ref.transform
        ref_crs = ref.crs
        height = ref.height
        width = ref.width

    if gdf.crs != ref_crs:
        print(f"[info] Reprojecting points from {gdf.crs} to {ref_crs}")
        gdf = gdf.to_crs(ref_crs)

    # 3) Validate required attributes
    missing = [c for c in BAND_NAMES if c not in gdf.columns]
    if missing:
        raise KeyError(
            "Colonne mancanti nel GPKG legacy: " + ", ".join(missing)
            + "\nControlla il layer o rinomina le colonne."
        )

    # 4) Map points -> pixel rows/cols and fill bands
    rows, cols = rasterio.transform.rowcol(transform, gdf.geometry.x.values, gdf.geometry.y.values)
    rows = np.asarray(rows, dtype=np.int32)
    cols = np.asarray(cols, dtype=np.int32)

    inside = (rows >= 0) & (rows < height) & (cols >= 0) & (cols < width)
    if not np.any(inside):
        raise RuntimeError("Nessun punto cade dentro l'estensione del raster di riferimento.")

    rows = rows[inside]
    cols = cols[inside]
    gdf = gdf.loc[inside].copy()

    # Initialize all output bands as float32 + NaN (converted to nodata on write)
    band_arrays = {name: np.full((height, width), np.nan, dtype=np.float32) for name in BAND_NAMES}

    print("[3/5] Filling raster cells from point attributes...")
    for name in BAND_NAMES:
        values = np.asarray(gdf[name], dtype=np.float32)
        band_arrays[name][rows, cols] = values

    # Optional coercion for ID-like bands: keep as float values but sanitize negatives
    for id_band in ["best_reseller_id_walk", "best_reseller_id_car"]:
        arr = band_arrays[id_band]
        arr = np.where(np.isfinite(arr) & (arr >= 0), arr, np.nan).astype(np.float32)
        band_arrays[id_band] = arr

    # 5) Write multi-band GeoTIFF in the exact 4.2 order
    print(f"[4/5] Writing fake output TIFF: {output_tif_path}")
    out_profile = ref_profile.copy()
    out_profile.update(dtype="float32", count=len(BAND_NAMES), nodata=NODATA_FLOAT, compress="lzw")

    with rasterio.open(output_tif_path, "w", **out_profile) as dst:
        for i, name in enumerate(BAND_NAMES, start=1):
            out = np.where(np.isfinite(band_arrays[name]), band_arrays[name], NODATA_FLOAT).astype(np.float32)
            dst.write(out, i)
            dst.set_band_description(i, name)

    n_input = len(gdf)
    n_written = int(np.sum(np.isfinite(band_arrays[BAND_NAMES[0]])))
    print("[5/5] Done")
    print(f"Input points used: {n_input:,}")
    print(f"Filled cells (band 1): {n_written:,}")


# ============================================================
# RUN
# ============================================================
DATA_DIR = pick_data_dir(DATA_DIR_CANDIDATES)
LEGACY_GPKG = f"{DATA_DIR}/{TRASH_SUBDIR}/{LEGACY_GPKG_NAME}"
REFERENCE_RASTER = f"{DATA_DIR}/{REFERENCE_RASTER_NAME}"
OUTPUT_TIF = f"{DATA_DIR}/{OUTPUT_TIF_NAME}"

build_fake_raster_from_points(
    gpkg_path=LEGACY_GPKG,
    ref_raster_path=REFERENCE_RASTER,
    output_tif_path=OUTPUT_TIF,
    layer_name=LEGACY_LAYER_NAME,
)

print(f"\nSaved fake raster: {OUTPUT_TIF}")

[1/5] Reading legacy points: dataset_big/trash/huff_preferred_distributor_per_pixel.gpkg | layer=pixel_preference
[2/5] Reading reference raster: dataset_big/Population.tif
[3/5] Filling raster cells from point attributes...
[4/5] Writing fake output TIFF: dataset_big/huff_preferred_distributor_per_pixel_FAKE.tif
[5/5] Done
Input points used: 563,482
Filled cells (band 1): 563,482

Saved fake raster: dataset_big/huff_preferred_distributor_per_pixel_FAKE.tif
